<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/theft_detection_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#from google.colab import files

#uploaded = files.upload()

Saving shoplifting_detection_final.keras to shoplifting_detection_final.keras


In [6]:
import os

print(os.listdir("/content"))
!ls -lh /content/shoplifting_detection_final.keras

['.config', '.ipynb_checkpoints', 'Shoplifting_1.mp4', 'Normal_1.mp4', 'shoplifting_detection_final.keras', 'sample_data']
-rw-r--r-- 1 root root 14M Sep  9 11:05 /content/shoplifting_detection_final.keras


In [4]:
import tensorflow as tf
import cv2
import numpy as np
import os

model = tf.keras.models.load_model("/content/shoplifting_detection_final.keras")

print("Model loaded!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

Model loaded!
Input shape: (None, 32, 224, 224, 3)
Output shape: (None, 2)


In [12]:
import cv2
import numpy as np

# -----------------------------
# SETTINGS
# -----------------------------
#VIDEO_PATH = "/content/Normal_1.mp4"
VIDEO_PATH = "/content/Shoplifting_1.mp4"
OUTPUT_PATH = "/content/Shoplifting_result.mp4"

FRAME_COUNT = 32
IMG_SIZE = 224

# Change this if your class labels are reversed
# Example:
# 0 = Normal
# 1 = Shoplifting
NORMAL_CLASS = 0
SHOPLIFTING_CLASS = 1


# -----------------------------
# OPEN VIDEO
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise Exception("Could not open video")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Resolution:", width, "x", height)
print("Total frames:", total_frames)


# -----------------------------
# OUTPUT VIDEO
# -----------------------------
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)


# -----------------------------
# FRAME BUFFER
# -----------------------------
frames = []

prediction_text = "Analyzing..."
confidence = 0


# -----------------------------
# PROCESS VIDEO
# -----------------------------
while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Keep original frame for displaying
    display_frame = frame.copy()

    # Convert BGR → RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Resize
    resized = cv2.resize(
        rgb_frame,
        (IMG_SIZE, IMG_SIZE)
    )

    # Normalize
    resized = resized.astype(np.float32) / 255.0

    # Add frame to buffer
    frames.append(resized)

    # ---------------------------------
    # When we have 32 frames
    # ---------------------------------
    if len(frames) == FRAME_COUNT:

        # Convert to numpy array
        input_data = np.array(frames)

        # Add batch dimension
        input_data = np.expand_dims(
            input_data,
            axis=0
        )

        # Prediction
        prediction = model.predict(
            input_data,
            verbose=0
        )

        binary_predictions = np.argmax(prediction, axis=1)

        probability = float(prediction[0][0])

        # ---------------------------------
        # CLASSIFICATION
        # ---------------------------------

        if binary_predictions == 1:

            prediction_text = "SHOPLIFTING"
            confidence = probability * 100

        else:

            prediction_text = "NORMAL"
            confidence = (1 - probability) * 100

        # Sliding window
        frames.pop(0)


FPS: 30.0
Resolution: 640 x 480
Total frames: 311


In [15]:


    # ---------------------------------
    # DISPLAY RESULT
    # ---------------------------------

    if prediction_text == "SHOPLIFTING":

        text = f"THEFT DETECTED - {confidence:.1f}%"

    else:

        text = f"NORMAL - {confidence:.1f}%"


    # Put text on video
    cv2.putText(
        display_frame,
        text,
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 0, 255) if prediction_text == "SHOPLIFTING"
        else (0, 255, 0),
        3
    )


    # Write frame
    out.write(display_frame)


# -----------------------------
# RELEASE
# -----------------------------
cap.release()
out.release()

print("Finished!")
print("Output saved as:", OUTPUT_PATH)

Finished!
Output saved as: /content/Shoplifting_result.mp4
